In [1]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

MODEL_NAME = "Qwen/Qwen3-4B"

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True
)

model.eval()

print("Base Qwen3 loaded successfully!")

config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

Base Qwen3 loaded successfully!


In [2]:
def build_story_prompt(title, text):

    prompt = f"""
You are a professional fiction writer.

Transform the following news article into ONE
creative, engaging, and coherent fictional story.

IMPORTANT RULES:
- Base the story only on this article.
- Preserve the main event and meaning.
- Do not combine this article with other articles.
- Connect the information naturally through characters and plot.
- Do not introduce unrelated major events.
- Write only the fictional story.
- Do not explain your process.
- Do not mention these instructions.

ARTICLE TITLE:
{title}

ARTICLE:
{text}

STORY:
"""

    return prompt

In [3]:
def generate_story(title, text, max_new_tokens=500):

    prompt = build_story_prompt(title, text)

    messages = [
        {
            "role": "user",
            "content": prompt
        }
    ]

    formatted_prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False
    )

    inputs = tokenizer(
        formatted_prompt,
        return_tensors="pt"
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.7,
            do_sample=True,
            top_p=0.9,
            repetition_penalty=1.1,
            no_repeat_ngram_size=3
        )

    generated_tokens = outputs[0][inputs["input_ids"].shape[1]:]

    story = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    )

    return story

In [4]:
title = "Football club fires manager after three matchdays"

text = """
The football club unexpectedly fired its manager
after only three matchdays following three consecutive
poor performances.
"""

story = generate_story(
    title,
    text
)

print(story)

The clock struck midnight, and the stadium lights flickered out, leaving only the faint glow of the city behind. Inside the boardroom, tension hung heavier than the air. The board had convened without warning, their faces etched with concern. Three games, three losses, and a season that looked set to spiral downward. The manager stood at the head of the table, his hands clasped tightly together, his jaw tight with unspoken words.

He had been hired with high hopes—promising leadership, a fresh vision for the team. But the players had struggled under his command. The defense crumbled, the midfield lost cohesion, and in the final third, the attackers found themselves stranded. Each loss was a blow, each goal a reminder of what was missing.

At the end of the third matchday, the board made their decision. It wasn’t a surprise, but it was sudden. The message was clear: the manager’s time was up. He stood up slowly, his eyes scanning the room. There was no anger, no defiance—only resignatio